# Some aspects of performance in image processing

Image processing on scientific data can take a long time, especially in settings where a series of images are acquired (time series for example). It is sometimes desirable to reduce the run-time of a given image processing pipeline, for example to make real-time image processing possible. 

In this tutorial, we cover several aspects of this specific task.

## Measuring computing times and profiling

If you want to optimize execution time, you first need to measure it. 

In [ ]:
from skimage import data
import numpy as np
img = data.eagle()
print(img.shape)

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(img, cmap='gray')

In [ ]:
%%timeit
from skimage import filters
img_median = filters.median(img, footprint=np.ones((7, 7), dtype=bool))

The %%timeit magic has many options (see https://ipython.readthedocs.io/en/stable/interactive/magics.html#magic-timeit). It is also possible to use the ``timeit`` module.

In [ ]:
from time import time
from skimage import filters
t1 = time()
img_median = filters.median(img, footprint=np.ones((7, 7), dtype=bool))
t2 = time()
print(t2 - t1)

Now we know the time taken by a given function, but we don't know why it should take a long time to execute. We can use Python's line profile to dive into the inner code of a function, and to identify which lines of the function take longer to execute.

In [ ]:
from skimage import color
img = color.rgb2gray((data.retina()[300:-300, 300:-300]))
out = filters.hessian(img)

In [ ]:
fig, ax = plt.subplots(1, 2)
ax[0].imshow(img)
ax[1].imshow(out)

In [ ]:
from line_profiler import LineProfiler, show_func
profile = LineProfiler()
profile.add_function(filters.hessian)

In [ ]:
out = profile.runcall(filters.hessian, img)

In [ ]:
profile.print_stats()

**Exercise**: what is the function taking the largest part of the execution time inside ``hessian``? Add this function to the line profiler and line-profile this other function while executing ``hessian``.

It is of course possible to apply the line profiler to your own functions. This can be useful when defining for example an image processing pipeline with a succession of operations. 

In [ ]:
%load_ext line_profiler

In [ ]:
from skimage import data
from skimage.filters import threshold_otsu
from skimage.segmentation import clear_border
from skimage.measure import label, regionprops
from skimage.morphology import closing, square
from skimage.color import label2rgb

def get_labels(img):
    # apply threshold
    thresh = threshold_otsu(image)
    bw = closing(image > thresh, square(5))
    # remove artifacts connected to image border
    cleared = clear_border(bw)
    # label image regions
    label_image = label(cleared)
    return label_image

In [ ]:
image = data.coins()

labels = get_labels(img)
plt.imshow(labels)

In [ ]:
%lprun -f get_labels get_labels(img)

scikit-image uses a lot of cython code, and it is also possible to profile cython code. Please see https://cython.readthedocs.io/en/latest/src/tutorial/profiling_tutorial.html for more information.

## Parallelizing execution by chunking the image array

One way to speed up the execution is to use the different CPU cores. Most scikit-image functions use only one core, so the other cores are not working. If we have several images, we can use different cores for the different images (using for example ``joblib.Parallel`` or ``dask``). But we can also divide the image into different sub-images and execute the function on the different sub-images. This is done by ``skimage.util.apply_parallel``.

**Warning**: most functions have boundary effects. Therefore you must use some overlap between the different sub-images in order to avoid artifacts.

In [ ]:
img = data.eagle()
print(img.shape)

In [ ]:
%%timeit -n 1 -r 1
from time import time
out = filters.median(img, np.ones((11, 11)))

In [ ]:
import joblib
joblib.cpu_count()

We need to install the optional dependency dask which is used by ``apply_parallel``.

In [ ]:
!pip install dask

In [ ]:
%%timeit -n 1 -r 1
from skimage import util
out = util.apply_parallel(filters.median, img, depth=7, extra_arguments=[np.ones((11, 11))])

In [ ]:
%%timeit -n 1 -r 1
from skimage import util
out = util.apply_parallel(filters.median, img, chunks=500, depth=7, extra_arguments=[np.ones((11, 11))])

**Exercise**: try different sizes of chunks and compare the execution times.

## Changing the algorithm

Often, the biggest performance gains are obtained by using a different algorithm. There is often a tradeoff between execution time and algorithm specificity, as exemplified by the example below about denoising. (Example adapted from https://scikit-image.org/docs/stable/auto_examples/filters/plot_denoise.html#sphx-glr-auto-examples-filters-plot-denoise-py).

In [ ]:
import matplotlib.pyplot as plt

from skimage.restoration import (denoise_tv_chambolle, denoise_bilateral,
                                 denoise_wavelet, denoise_nl_means, estimate_sigma)
from skimage import data, img_as_float
from skimage.util import random_noise
from time import time

original = img_as_float(data.chelsea()[100:250, 50:300])

sigma = 0.155
noisy = random_noise(original, var=sigma**2)

t0 = time()
img_tv = denoise_tv_chambolle(noisy, weight=0.1, channel_axis=-1)
t1 = time()
print(f"TV: {t1 - t0} s" )

t0 = time()
img_nlmeans = denoise_nl_means(noisy,
                channel_axis=-1)
t1 = time()
print(f"Non-local means: {t1 - t0} s")

t0 = time()
img_bil = denoise_bilateral(noisy, sigma_color=0.05, sigma_spatial=11,
                channel_axis=-1)
t1 = time()
print(f"Bilateral: {t1 - t0} s")


fig, ax = plt.subplots(nrows=1, ncols=4, figsize=(8, 5),
                       sharex=True, sharey=True)

plt.gray()

# Estimate the average noise standard deviation across color channels.
sigma_est = estimate_sigma(noisy, channel_axis=-1, average_sigmas=True)
# Due to clipping in random_noise, the estimate will be a bit smaller than the
# specified sigma.
print(f'Estimated Gaussian noise standard deviation = {sigma_est}')

ax[0].imshow(noisy)
ax[0].axis('off')
ax[0].set_title('Noisy')
ax[1].imshow(img_tv)
ax[1].axis('off')
ax[1].set_title('TV')
ax[2].imshow(img_nlmeans)
ax[2].axis('off')
ax[2].set_title('Non-local means')
ax[3].imshow(img_bil)
ax[3].axis('off')
ax[3].set_title('Bilateral')


fig.tight_layout()

plt.show()
